In [1]:
import sys
sys.path.insert(0, "..")
from src.spark_utils import get_spark
from pyspark.sql import functions as F

spark = get_spark("02-clean")
df = spark.read.parquet("../data/yellow_2016_03.parquet")
print(f"Loaded: {df.count():,} rows")

Loaded: 12,210,952 rows


## Problem

We predict **tip_amount** for credit-card trips only.

NYC TLC records cash tips as 0, regardless of what was actually given.
Mixing cash trips into training would teach the model that ~50% of all
trips have zero tip, when in reality we just don't see those tips.
This is a recording artifact, not a signal.

Filtering to payment_type=1 (credit card) makes the target honest at
the cost of halving the dataset. ~6M rows remain — still more than
enough for the scaling experiments.

In [2]:
df = df.withColumn(
    "trip_duration_min",
    (F.unix_timestamp("tpep_dropoff_datetime") -
     F.unix_timestamp("tpep_pickup_datetime")) / 60.0
)

In [3]:
steps = []
def step(name, frame):
    n = frame.count()
    steps.append((name, n))
    print(f"{name:35s} {n:>12,}")
    return frame

df = step("0. raw",                 df)
df = step("1. payment_type == 1",   df.filter(F.col("payment_type") == 1))
df = step("2. passenger 1..6",      df.filter(F.col("passenger_count").between(1, 6)))
df = step("3. distance 0..100 mi",  df.filter((F.col("trip_distance") > 0) & (F.col("trip_distance") < 100)))
df = step("4. fare 2.5..500",       df.filter(F.col("fare_amount").between(2.5, 500)))
df = step("5. tip 0..fare*2",       df.filter((F.col("tip_amount") >= 0) & (F.col("tip_amount") <= F.col("fare_amount") * 2)))
df = step("6. total > 2.5",         df.filter(F.col("total_amount") > 2.5))
df = step("7. duration 1..240 min", df.filter(F.col("trip_duration_min").between(1, 240)))
df = step("8. NYC bbox",            df.filter(
    F.col("pickup_longitude").between(-74.05, -73.70) &
    F.col("pickup_latitude").between(40.55, 40.95) &
    F.col("dropoff_longitude").between(-74.05, -73.70) &
    F.col("dropoff_latitude").between(40.55, 40.95)
))
df = step("9. March 2016 only",     df.filter(
    F.col("tpep_pickup_datetime").between("2016-03-01 00:00:00",
                                          "2016-03-31 23:59:59")
))

# Save the waterfall for the report
import pandas as pd
pd.DataFrame(steps, columns=["filter", "rows_remaining"]).to_csv(
    "../results/cleaning_waterfall.csv", index=False
)

0. raw                                12,210,952
1. payment_type == 1                   8,127,391
2. passenger 1..6                      8,126,958
3. distance 0..100 mi                  8,101,606
4. fare 2.5..500                       8,101,283
5. tip 0..fare*2                       8,099,501
6. total > 2.5                         8,099,501
7. duration 1..240 min                 8,074,846
8. NYC bbox                            7,937,904
9. March 2016 only                     7,937,904


In [4]:
df.select("trip_distance", "fare_amount", "tip_amount",
          "trip_duration_min", "total_amount") \
  .summary("min", "25%", "50%", "75%", "max").show()

+-------+-------------+-----------+----------+------------------+------------+
|summary|trip_distance|fare_amount|tip_amount| trip_duration_min|total_amount|
+-------+-------------+-----------+----------+------------------+------------+
|    min|         0.01|        2.5|       0.0|               1.0|         3.3|
|    25%|          1.1|        7.0|      1.35| 6.983333333333333|        9.36|
|    50%|          1.8|        9.5|       2.0|11.383333333333333|       12.95|
|    75%|         3.36|       15.0|       3.0|18.083333333333332|       18.96|
|    max|        95.74|      500.0|     200.0|             239.2|      527.59|
+-------+-------------+-----------+----------+------------------+------------+



In [5]:
(df.write
   .mode("overwrite")
   .parquet("../data/yellow_2016_03_clean.parquet"))
print(f"Cleaned dataset: {df.count():,} rows")
spark.stop()

Cleaned dataset: 7,937,904 rows
